# Read new data using Auto Loader
Script Purpose:
* Read new, unmodifed data 
* Add information about ingestion timestamp
* Write data to bronze layer

In [0]:
from pyspark.sql import functions as F

With the use of Databricks Auto Loader:
* Read csv files
* Keep Auto Loader's ingested data schema
* Keep headers from csv files
* Load data from the source (folder with generated data) 

In [0]:
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", "/Volumes/deliveries/default/checkpoint/schema")
    .option("header", "true")
    .load("/Volumes/deliveries/default/generated_fact_data/")
)

Add a timestamp of ingestion time

In [0]:
df = df.withColumn("ingested_at", F.current_timestamp())


Write stream into the table:
* Delta table to keep historic data
* Append mode to add new data to table
* Auto Loader's checkpoint location
* Trigger to stop process after code finishes the run
* Write DataFrame into bronze layer

In [0]:
(df.writeStream
.format("delta")
.outputMode("append")
.option("checkpointLocation", "/Volumes/deliveries/default/checkpoint")
.trigger(availableNow=True)
.toTable("deliveries.bronze.gen_shipments"))

See the outcome data table

In [0]:
df_bronze = spark.read.table("deliveries.bronze.gen_shipments")
df_bronze.limit(1).display()